In [5]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess

from preprocess import correct_inputs as ci
from utils.load_environmental_variables import *

import pandas as pd
from expression import build_me_model

dummy_protein = [True, False]
minimal_proteome = [True, False]
compress_mrna = [True, False]

counter = 0
res = pd.DataFrame(columns = ['dp', 'mp', 'cm', 'status'])
for dp in dummy_protein:
    for mp in minimal_proteome:
        for cm in compress_mrna:
            try:
                toy_me_model, builder = build_me_model.build_me(minimal_proteome = mp, compress_mrna = cm, 
                                                            dummy_protein = dp)
                sln, stat, _ = toy_me_model.solve_lp(mu_val =  1e-9)
                res.loc[counter, : ] = [dp, mp, cm, stat.max()]
            except:
                res.loc[counter, : ] = [dp, mp, cm, float('nan')]
            counter += 1
res.to_csv('trash.csv')

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


In [2]:
import pandas as pd
res = pd.read_csv('trash.csv')

In [3]:
res

,Unnamed: 0,dp,mp,cm,status
0,0,True,True,True,0.0
1,1,True,True,False,NaN
2,2,True,False,True,NaN
3,3,True,False,False,NaN
4,4,False,True,True,0.0
5,5,False,True,False,0.0
6,6,False,False,True,0.0
7,7,False,False,False,0.0


In [6]:
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                            dummy_protein = True)
sln, stat, _ = toy_me_model.solve_lp(mu_val =  1e-9)

Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


  1%|          | 4/591 [00:00<00:19, 29.82it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:23<00:00, 25.08it/s]


Generate protein expression reactions for expression module enzymes, this step may take a few minutes


...

Check reaction mass balances


  7%|▋         | 909/12723 [00:00<00:01, 9003.90it/s]

Check correct coupling of metabolic machinery


100%|██████████| 12723/12723 [00:07<00:00, 1698.98it/s]
../../../scripts/core/model.py:305 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Time to build: 5.65794153213501 minutes
Getting MINOS parameters...
Done in 245.162 seconds with status 0


In [ ]:
from expression import build_me_model
from uniform_processes import biomass
from utils import parameters as params
import copy
import pandas as pd
import numpy as np
from tqdm import tqdm
import multiprocessing
from core.reaction import ME_Reaction


def get_mod(frac):
    store = dict()
    m_id = 'dummy_{}'.format(frac) if frac is not None else 'no_dummy'

    model, builder = build_me_model.build_me(non_machinery = [], minimal_proteome = True, 
                   compress_mrna = False, unmodeled_protein_frac = frac, model_id = m_id)
    
    bad_rxn = list()
    for r in params.human_model.reactions:
        if len(r.genes) == 0:
            for r_ in model.reactions: 
                if isinstance(r_, ME_Reaction):
                    if (r_.id == r.id) or (r_.cobra_id == r.id):
                        if len({k for k,v in r_.coupled_metabolites.items() if 'HGNC:DUMMY' not in k.id}) > 0:
                            bad_rxn.append(r_.id)
                        break
    if len(bad_rxn) > 0
    

    models = [model]
    if frac is not None:
        sink_model = copy.deepcopy(model)
        sink_model.id = 'dummy_sink_{}'.format(frac)
        sink_model.add_boundary(sink_model.metabolites.get_by_id('HGNC:10419_folded_protein_c'), type = 'sink')
        models.append(sink_model)

    for model in models:
        store[model.id] = {'model': model, 'builder': builder}

        print('Solve ' + m_id)
        sln, stat, _ = store[model.id]['model'].solve_lp(mu_val = 1e-9)
        ir = store[model.id]['model'].infeasible_reactions(1e-9, sln, stat)

        store[model.id]['sln'] = sln
        store[model.id]['stat'] = stat
        store[model.id]['infeasible_reactions'] = ir

    return store

def get_mods(fracs, n_cores = 3):
    pool = multiprocessing.Pool(processes = n_cores)
    stores = pool.map(get_mod,fracs)
    pool.close()
    return stores

In [ ]:
stores = get_mods(fracs = [None, 0.001, params.unmodeled_protein_frac])

res = dict()
for store in stores:
    for k,v in store.items():
        res[k] = v
print(list(res.keys()))

In [ ]:
tol = np.max([abs(v) for v in res['no_dummy']['infeasible_reactions'].values()])

r_ids = []
for k_ in res:
    r_ids += [k for k,v in res[k_]['infeasible_reactions'].items() if abs(v) > tol]

r_ids = pd.Series([biomass.pb_reaction.id] + [r.id for r in biomass.biomass_reactions]  + r_ids).unique().tolist()
res_df = pd.DataFrame(columns = res.keys(), index = r_ids)    

for k in res.keys():
    for r_id in r_ids:
        try:
            res_df.loc[r_id, k] = res[k]['sln'][res[k]['model'].reactions.index(r_id)]
        except:
            res_df.loc[r_id, k] = float('nan')

fail = res_df[abs(res_df['dummy_0.879584658137385'] - res_df['dummy_sink_0.879584658137385']) > tol]

In [ ]:
model = res['no_dummy']['model']

In [ ]:
self = res['no_dummy']['model']
self.m_model = params.human_model.copy()
mismatch = check_coupling(self)